In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# Create SparkSession from builder
spark = SparkSession.builder.master("local[1]") \
                    .appName('PySpark ETL') \
                    .config("spark.jars.packages", "com.microsoft.sqlserver:mssql-jdbc:12.4.2.jre11") \
                    .config("spark.jars", "/opt/spark/jars/postgresql-42.5.2.jar") \
                    .getOrCreate()

print(spark.sparkContext)
print("Spark Version : "+ spark.version)
print("Spark App Name : "+ spark.sparkContext.appName)

<SparkContext master=local[1] appName=PySpark ETL>
Spark Version : 3.4.0
Spark App Name : PySpark ETL


Read SQL Server Table to PySpark DataFrame

In [3]:
# SQL Server connection parameters
server_name = "mssql-server"  # Docker container name (or localhost if running outside Docker)
database_name = "AdventureWorksDW2022"
username = "SA"
password = "StrongPassw0rd123"

# JDBC connection URL for SQL Server
url = f"jdbc:sqlserver://{server_name}:1433;database={database_name};user={username};password={password};trustServerCertificate=true"

print(f"Connection URL: {url}")

Connection URL: jdbc:sqlserver://mssql-server:1433;database=AdventureWorksDW2022;user=SA;password=StrongPassw0rd123;trustServerCertificate=true


In [4]:
# Read data from SQL Server using JDBC
sql_query = """select t.name as table_name
from sys.tables t 
where t.name in ('DimProduct', 'DimProductSubcategory', 'DimProductCategory', 'DimSalesTerritory', 'FactInternetSales')"""

df = spark.read.format("jdbc") \
    .option("url", url) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("query", sql_query) \
    .load()

# Display the dataframe
print(f"Total rows: {df.count()}")
df.show(truncate=False)
df.printSchema()

Total rows: 5
+---------------------+
|table_name           |
+---------------------+
|DimProduct           |
|DimProductCategory   |
|DimProductSubcategory|
|DimSalesTerritory    |
|FactInternetSales    |
+---------------------+

root
 |-- table_name: string (nullable = true)



In [18]:
# Target PostgreSQL connection parameters
server="postgres"
target_db="postgres_db"
uid="postgres"
pwd="postgres"
target_url = f"jdbc:postgresql://{server}:5432/{target_db}?user={uid}&password={pwd}"
target_driver = "org.postgresql.Driver"

In [19]:
def load(df, tbl):
    try:
        rows_imported = 0
        print(f'importing rows {rows_imported} to {rows_imported + df.count()}... for table {tbl}')
        df.write.mode("overwrite") \
        .format("jdbc") \
        .option("url", target_url) \
        .option("user", uid) \
        .option("password", pwd) \
        .option("driver", target_driver) \
        .option("dbtable", "src_" + tbl) \
        .save()
        print("Data imported successful")
        rows_imported += df.count()
    except Exception as e:
        print("Data load error: " + str(e))

In [20]:
def extract():
    try:
        df = spark.read.format("jdbc") \
            .option("url", url) \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .option("query", sql_query) \
            .load()
        # get table names
        data_collect = df.collect()
        # looping thorough each row of the dataframe
        for row in data_collect:
        # while looping through each
            print(row["table_name"])
            tbl_name = row["table_name"]
            df = spark.read \
            .format("jdbc") \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .option("url", url) \
            .option("dbtable", f"dbo.{tbl_name}") \
            .load()
            # print(df.show(5))
            load(df, tbl_name)
            print("Data loaded successfully")
    except Exception as e:
        print("Data extract error: " + str(e))

In [21]:
extract()

DimProduct
importing rows 0 to 606... for table DimProduct
Data imported successful
Data loaded successfully
DimProductCategory
importing rows 0 to 4... for table DimProductCategory
Data imported successful
Data loaded successfully
DimProductSubcategory
importing rows 0 to 37... for table DimProductSubcategory
Data imported successful
Data loaded successfully
DimSalesTerritory
importing rows 0 to 11... for table DimSalesTerritory
Data imported successful
Data loaded successfully
FactInternetSales
importing rows 0 to 60398... for table FactInternetSales
Data imported successful
Data loaded successfully


In [22]:
spark.sparkContext.stop()